# TFL Project

## Step 1: Data Preparation
- Import data.
- Merge datasets into one dataframe.
- Format data correctly as we want it.
- Create the necessary variables.
- Check the data

In [1]:
# Imports for Step 1
import pandas as pd

### Step 1A: Import Data

In [2]:
# Import the datasets
df2019 = pd.read_csv("StationFootfall_2019.csv")
df2020 = pd.read_csv("StationFootfall_2020.csv")
df2021 = pd.read_csv("StationFootfall_2021.csv")
df2022 = pd.read_csv("StationFootfall_2022.csv")
df2023 = pd.read_csv("StationFootfall_2023.csv")
df2024 = pd.read_csv("StationFootfall_2024.csv")
df2025 = pd.read_csv("stationfootfall-2025.csv")

### Step 1B: Merge Datasets

In [3]:
# Ready the data to be merged
df2019 = df2019.rename(columns={"DayOFWeek": "DayOfWeek"})
df2020 = df2020.rename(columns={"DayOFWeek": "DayOfWeek"})
df2021 = df2021.rename(columns={"DayOFWeek": "DayOfWeek"})
df2022 = df2022.rename(columns={"DayOFWeek": "DayOfWeek"})

In [4]:
# Merge the dataset
df = pd.concat([
    df2019,
    df2020,
    df2021,
    df2022,
    df2023,
    df2024,
    df2025
], ignore_index=True)

### Step 1C: Format Data

In [5]:
# Format variable names
df = df.rename(columns = {'TravelDate': 'date'})
df = df.rename(columns = {'DayOfWeek': 'day'})
df = df.rename(columns = {'Station': 'station'})
df = df.rename(columns = {'EntryTapCount': 'enter_count'})
df = df.rename(columns = {'ExitTapCount': 'exit_count'})

In [6]:
# Format the time variable
df["date"] = pd.to_datetime(df["date"], format="%Y%m%d")

In [7]:
# Fixing some issues with the naming convention of stations in the data
df["station"] = df["station"].replace("Custom House EL", "Custom House Elizabeth Line")
df["station"] = df["station"].replace("Canary Wharf EL", "Canary Wharf Elizabeth Line")
df["station"] = df["station"].replace("Cannon Street ", "Cannon Street")
df["station"] = df["station"].replace("Woolwich EL", "Woolwich Elizabeth Line")


### Step 1D: Create necessary variables

In [8]:
# Create total variable
df["total_count"] = df["enter_count"] + df["exit_count"]

In [9]:
# Create year and month variables
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month

In [10]:
# Create weekend variable
df["is_weekend"] = df["day"].isin(
    ["Saturday", "Sunday"]
)

In [11]:
# Create the month name variable
df["month_name"] = pd.to_datetime(
    df["month"], format="%m"
).dt.month_name()

In [12]:
# Create a COVID variable
df["covid_period"] = df["year"].apply(
    lambda x: "Pre-COVID" if x == 2019
    else "COVID" if x in [2020, 2021]
    else "Post-COVID"
)

In [13]:
# Add a boroughs column
df_sl = pd.read_csv('station_lookup.csv')

df = df.merge(df_sl, on="station", how="left")

In [14]:
# Drop if missing borough variable as these are usually outside of London
df = df.dropna(subset=['borough'])

### Step 1D: Check the data 

In [15]:
# Check for duplicates
df.duplicated(
    subset=["date", "station"]
).sum()

np.int64(0)

In [16]:
# Check extreme values
(df[["enter_count", "exit_count", "total_count"]] < 0).sum()

enter_count    0
exit_count     0
total_count    0
dtype: int64

In [17]:
# Check years
df["year"].unique()

array([2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026], dtype=int32)

## Step 2: Exploratory Statistics
- Generate Descriptive statistics
- Generate key facts about the datasets
- Visualise these facts

### Step 2A: Generate Descriptive Statistics

In [18]:
df[[ # descrptive statistics
    "enter_count",
    "exit_count",
    "total_count"
]].describe()

,enter_count,exit_count,total_count
count,1.076875e+06,1.076875e+06,1.076875e+06
mean,8.170785e+03,8.200007e+03,1.637079e+04
std,1.324272e+04,1.358774e+04,2.679683e+04
min,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.895000e+03,1.825000e+03,3.730000e+03
50%,4.061000e+03,3.960000e+03,8.024000e+03
75%,8.414000e+03,8.345000e+03,1.676400e+04
max,1.614980e+05,1.805380e+05,3.299810e+05


### Step 2B: Generate Key facts

In [19]:
# Top 10 most popular station across our whole dataset
df.groupby('station')['total_count'].sum().sort_values(ascending=False).head(10)

station
Kings Cross St Pancras    481508056
Victoria                  417222116
Waterloo                  380627273
London Bridge             380583303
Stratford                 373111069
Oxford Circus             366423120
Liverpool Street          366029199
Tottenham Court Road      317876745
Bond Street               234688786
Paddington                219757274
Name: total_count, dtype: int64

In [20]:
# Bottom 10 most popular station across our whole dataset
df.groupby('station')['total_count'].sum().sort_values(ascending=True).head(10)

station
Emerson Park             1289291
Beckton Park             1971455
South Hampstead          2685946
Headstone Lane           2805412
Barking Riverside        3425663
Penge West               3635146
Stratford High Street    3658553
Abbey Road DLR           3739565
Wandsworth Road          3741703
Stamford Hill            3771315
Name: total_count, dtype: int64

In [21]:
# Top 10 daily average station across our whole dataset
df.groupby("station")["total_count"].mean().sort_values(ascending=False).head(10)

station
Kings Cross St Pancras    175861.233017
Victoria                  152437.747899
London Bridge             139102.084430
Waterloo                  138965.780577
Stratford                 136371.004751
Oxford Circus             133779.890471
Liverpool Street          133587.298905
Tottenham Court Road      116055.766703
Bond Street                85746.724881
Paddington                 80232.666667
Name: total_count, dtype: float64

In [22]:
# Bottom 10 daily average station across our whole dataset
df.groupby("station")["total_count"].mean().sort_values(ascending=True).head(10)

station
Emerson Park              492.659916
Beckton Park              729.628053
South Hampstead          1019.334345
Headstone Lane           1062.656061
Penge West               1358.933084
Stratford High Street    1363.097243
Abbey Road DLR           1377.372007
Stamford Hill            1417.254791
North Ealing             1462.830527
Wandsworth Road          1468.486264
Name: total_count, dtype: float64

In [23]:
# Most taps per month
df.groupby("month_name")["total_count"].mean().sort_values(ascending=False).head(3)

month_name
October     17963.649663
November    17875.202173
February    17493.737277
Name: total_count, dtype: float64

In [24]:
# Least taps per month
df.groupby("month_name")["total_count"].mean().sort_values(ascending=True).head(3)

month_name
April     15036.880136
August    15279.730171
May       15704.213081
Name: total_count, dtype: float64

In [25]:
# Most taps per day
df.groupby("day")["total_count"].mean().sort_values(ascending=False).head(3)

day
Thursday     18713.495300
Wednesday    18301.764211
Tuesday      17932.716681
Name: total_count, dtype: float64

In [26]:
# Least taps per day
df.groupby("day")["total_count"].mean().sort_values(ascending=True).head(3)

day
Sunday      10653.751361
Saturday    15200.549571
Monday      15911.983491
Name: total_count, dtype: float64

In [27]:
# Most taps per borough
df.groupby("borough")["total_count"].sum().sort_values(ascending=False).head(10)

borough
Westminster               3173276607
Camden                    2018149235
City of London            1303358057
Tower Hamlets             1003640832
Lambeth                    974006455
Newham                     919793239
Southwark                  805937110
Kensington and Chelsea     717773074
Hammersmith and Fulham     693951142
Islington                  653428239
Name: total_count, dtype: int64

In [28]:
# Least taps per borough
df.groupby("borough")["total_count"].sum().sort_values(ascending=True).head(10)

borough
Bromley                           26313334
Croydon                           50830358
Richmond upon Thames              99461181
Islington / Haringey (border)    124280927
Enfield                          144173354
Hounslow                         153810558
Havering                         158380187
Lewisham                         171794381
Barking and Dagenham             200404775
Harrow                           226200408
Name: total_count, dtype: int64

In [35]:
df.groupby("borough")["total_count"].mean().sort_values(ascending=False).head(10)

borough
Islington / Haringey (border)    45424.315424
City of London                   40044.182653
Westminster                      38567.272415
Lambeth                          30052.652114
Camden                           27783.273930
Southwark                        27307.868058
Wandsworth                       22264.953071
Kensington and Chelsea           22053.432697
Greenwich                        20265.437846
Islington                        20196.211875
Name: total_count, dtype: float64

In [36]:
df.groupby("borough")["total_count"].mean().sort_values(ascending=True).head(10)

borough
Bromley       3244.554131
Enfield       5361.398014
Hillingdon    6243.076549
Harrow        6440.786105
Lewisham      7011.729358
Hounslow      7100.805965
Havering      7349.428631
Ealing        7723.794587
Hackney       7800.163171
Brent         8422.695822
Name: total_count, dtype: float64

### Step 3: By Station Analysis

In [31]:
# Total taps per year by station
annual_df = (df.groupby(["station", "year"])["total_count"].sum().unstack())

In [32]:
# Add the boroughs column to the annual data
annual_df = (annual_df.reset_index().merge(df_sl, on="station", how="left"))

In [33]:
# Growth variable
annual_df["growth_pct"] = ((annual_df[2025] - annual_df[2019]) / annual_df[2019]) * 100

In [34]:
# Higehst growth in demand for station usage from 2019 to 2025 (All Lizzy line stations)
annual_df.sort_values("growth_pct", ascending=False).head(10)

,station,2019,2020,2021,2022,2023,2024,2025,2026,borough,growth_pct
3,Acton Main Line,303004.0,162829.0,274219.0,877171.0,2034988.0,2533196.0,2809853.0,1523514.0,Ealing,827.331982
158,Heathrow T2&3 TfL Rail/HEx,828561.0,239270.0,274775.0,2089628.0,4948106.0,5896284.0,6587186.0,3665794.0,Hillingdon,695.015213
160,Heathrow T5 TfL Rail/HEx,491840.0,201329.0,231236.0,1220010.0,2748389.0,3434918.0,3837321.0,2062946.0,Hillingdon,680.197015
159,Heathrow T4 TfL Rail/HEx,381401.0,78435.0,315.0,321925.0,1206686.0,1665358.0,1878535.0,1054908.0,Hillingdon,392.535415
148,Hanwell,398291.0,220430.0,411862.0,807685.0,1346081.0,1548931.0,1687025.0,932236.0,Ealing,323.565935
1,Abbey Wood,3274465.0,1687009.0,1959639.0,4613529.0,8610234.0,10439767.0,11667552.0,6141261.0,Greenwich,256.319338
224,Maryland,1276017.0,759733.0,1198636.0,1880992.0,3107610.0,3758137.0,4011495.0,2078686.0,Newham,214.376297
313,Southall,2769700.0,1445936.0,2114143.0,3628793.0,5837635.0,6812679.0,7324050.0,3696569.0,Ealing,164.434776
270,Pudding Mill Lane,640759.0,301318.0,377203.0,1133456.0,1824649.0,1849783.0,1670091.0,972005.0,Newham,160.642613
374,West Ealing,1012113.0,464961.0,653067.0,1169475.0,2036837.0,2416200.0,2536624.0,1355204.0,Ealing,150.626560


### Step 4: By Borough Analysis

In [37]:
# Total taps per year by borough
annual_bor_df = (df.groupby(["borough", "year"])["total_count"].sum().unstack())

In [38]:
# Growth variable
annual_bor_df["growth_pct"] = ((annual_bor_df[2025] - annual_bor_df[2019]) / annual_bor_df[2019]) * 100

In [40]:
# Higehst growth in demand for booughs usage from 2019 to 2025 (All Lizzy line stations)
annual_bor_df.sort_values("growth_pct", ascending=False).head(5)

year,2019,2020,2021,2022,2023,2024,2025,2026,growth_pct
borough,,,,,,,,,
Greenwich,50774173,22111888,26878977,43908107,58160988,61621126,61229241,30548360,20.591311
Hillingdon,51335182,19636514,21684644,39791338,52854138,57516153,58806847,29495478,14.554667
Havering,23850584,11230989,14085661,19680943,23668477,25388820,26718543,13756170,12.024691
Newham,143991887,66027101,79403746,116747756,140708364,148666527,148782948,75464910,3.327313
Wandsworth,69939940,30552833,36308581,55454981,69656972,72229354,72056920,36449951,3.026854
